# 完整的业务解决方案

## 现在我们将把我们的项目从第一天提升到一个新的水平

### 业务挑战：

创建一个产品，为公司制作宣传册，供潜在客户、投资者和潜在员工使用。

我们将获得公司名称及其主要网站。

有关实际业务应用程序的示例，请参阅本笔记本的末尾。

请记住：如果您有问题或想法，我随时乐意为您服务！请务必伸出援手。

In [ ]:
# 进口
# 如果这些失败，请检查您是否在命令提示符中使用 (llms) 从“已激活”环境运行

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [ ]:
# 初始化和常量

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

In [ ]:
links = fetch_website_links("https://edwarddonner.com")
links

## 第一步：让 GPT-5-nano 找出哪些链接是相关的

### 使用 gpt-5-nano 调用来读取网页上的链接，并以结构化 JSON 进行响应。  
它应该决定哪些链接是相关的，并将相关链接（例如“/about”）替换为“https://company.com/about”。  
我们将使用“一次性提示”，其中我们提供了一个示例，说明它应该如何在提示中做出响应。

对于法学硕士来说，这是一个很好的用例，因为它需要细致入微的理解。想象一下，尝试在没有法学硕士的情况下通过解析和分析网页来编写此代码 - 这将非常困难！

旁注：有一种更先进的技术称为“结构化输出”，其中我们要求模型根据规范进行响应。我们将在第 8 周的自主 Agentic AI 项目中介绍这项技术。

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://edwarddonner.com"))

In [ ]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://huggingface.co")

## 第二步：制作宣传册！

将所有详细信息组装到 GPT-5-nano 的另一个提示中

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# 或者取消注释下面的行以获得更幽默的小册子 - 这表明合并“语气”是多么容易：

# 宣传册系统提示=“”“
# 您是一名助理，负责分析公司网站上几个相关页面的内容
# 并为潜在客户、投资者和新员工制作了一本关于公司的简短、幽默、有趣、诙谐的宣传册。
# 在没有代码块的情况下以 Markdown 方式进行响应。
# 如果您有信息，请包括公司文化、客户和职业/工作的详细信息。
# """



In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

In [ ]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("HuggingFace", "https://huggingface.co")

## 最后 - 一个小改进

通过小的调整，我们可以更改此设置，以便结果从 OpenAI 传回，
熟悉的打字机动画

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
# 尝试制作抱脸小册子时将系统提示改为幽默版：

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="宽度：150px；高度：150px；垂直对齐：中间；">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">商业应用</h2>
            <span style="color:#181;">在本练习中，我们扩展了 Day 1 代码以进行多个 LLM 调用并生成文档。

这可能是 Agentic AI 设计模式的第一个示例，因为我们结合了对 LLM 的多次调用。这将在第 2 周进行更多介绍，然后我们将在第 8 周大规模回归 Agentic AI，届时我们将构建完全自主的 Agent 解决方案。

以这种方式生成内容是最常见的用例之一。与总结一样，这可以应用于任何垂直业务。编写营销内容、根据规范生成产品教程、创建个性化电子邮件内容等等。探索如何将内容生成应用到您的业务中，并尝试为自己制作一个概念验证原型。看看其他学生在社区贡献文件夹中做了什么——这么多有价值的项目——太疯狂了！</span>
        </td>
    </tr>
</表>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="宽度：150px；高度：150px；垂直对齐：中间；">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">在进入第 2 周之前（这非常有趣）</h2>
            <span style="color:#900;">请参阅第 1 周练习笔记本，了解第 1 周结束时的挑战。这将为您提供一些使用 Frontier API 的基本练习，并为第 2 周做好准备。</span>
        </td>
    </tr>
</表>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="宽度：150px；高度：150px；垂直对齐：中间；">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">关于 3 个有用资源的提醒</h2>
            <span style="color:#f71;">1.本课程的资源可在<a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">此处获取。</a><br/>
            2. 我在 LinkedIn <a href="https://www.linkedin.com/in/eddonner/">此处</a>，我喜欢与参加该课程的人联系！<br/>
            3. 我正在尝试 X/Twitter，我在 <a href="https://x.com/edwarddonner">@edwarddonner<a> 上，希望人们能教我如何操作。  
            </span>
        </td>
    </tr>
</表>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="宽度：150px；高度：150px；垂直对齐：中间；">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">终于！我有一个特别的请求给你</h2>
            <span style="color:#090;">
                我的编辑告诉我，学生在 Udemy 上对这门课程进行评分会产生巨大的影响 - 这是 Udemy 决定是否向其他人展示该课程的主要方式之一。如果您能花一点时间评价一下，我将非常感激！无论如何，如果我可以随时提供帮助，请随时通过 ed@edwarddonner.com 与我联系。
            </span>
        </td>
    </tr>
</表>